# Project 2: Distributional Hypothesis

**Course:** Deep Learning and Natural Language Processing (PhDAI-831-A01)  
**University:** University of the Cumberlands  

## Objective

This project demonstrates the distributional hypothesis by normalizing a small collection of sentences, extracting words that occur within a specified distance of a target word, and representing the target word as a distributional vector based on its surrounding context.

In [1]:
# Environment Verification

import sys
import platform

print("Python version :", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.system(), platform.release())

Python version : 3.11.9
Python executable: c:\Users\dubey\Documents\UC_GitLab\.venv_global\Scripts\python.exe
Operating system: Windows 10


## 1. Imports and Sample Sentences

This section imports the required Python libraries and defines a small sample corpus that will be used to demonstrate the distributional hypothesis.

In [2]:
# Code Block 1: Imports and Sample Sentences

import re
from collections import Counter

sample_sentences = [
    "The dog chased the cat.",
    "The dog played with the ball.",
    "The cat chased the mouse.",
    "The cat played with the toy.",
    "The dog chased the mouse."
]

print("Number of sample sentences:", len(sample_sentences))

for i, sentence in enumerate(sample_sentences, start=1):
    print(f"Sentence {i}: {sentence}")

Number of sample sentences: 5
Sentence 1: The dog chased the cat.
Sentence 2: The dog played with the ball.
Sentence 3: The cat chased the mouse.
Sentence 4: The cat played with the toy.
Sentence 5: The dog chased the mouse.


## 2. Text Normalization

Before extracting context words, the sentences are normalized so that capitalization and punctuation do not cause the same word to be treated as different tokens.

The normalization process performs three operations:

1. Converts all text to lowercase.
2. Removes punctuation.
3. Splits the cleaned sentence into individual word tokens.

In [3]:
# Code Block 2: Text Normalization Function

def normalize_text(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^\w\s]", "", sentence)
    tokens = sentence.split()
    return tokens


# Test the normalization function on the first sentence
example_sentence = sample_sentences[0]
normalized_example = normalize_text(example_sentence)

print("Original sentence  :", example_sentence)
print("Normalized tokens  :", normalized_example)

Original sentence  : The dog chased the cat.
Normalized tokens  : ['the', 'dog', 'chased', 'the', 'cat']


## 3. Normalize All Sample Sentences

The normalization function is now applied to every sentence in the sample corpus. The result is a list of tokenized sentences that will be used for context extraction.

In [4]:
# Code Block 3: Normalize the Full Corpus

normalized_sentences = [
    normalize_text(sentence)
    for sentence in sample_sentences
]

for i, tokens in enumerate(normalized_sentences, start=1):
    print(f"Sentence {i}: {tokens}")

Sentence 1: ['the', 'dog', 'chased', 'the', 'cat']
Sentence 2: ['the', 'dog', 'played', 'with', 'the', 'ball']
Sentence 3: ['the', 'cat', 'chased', 'the', 'mouse']
Sentence 4: ['the', 'cat', 'played', 'with', 'the', 'toy']
Sentence 5: ['the', 'dog', 'chased', 'the', 'mouse']


## 4. Extract Context Words Within Distance d

For a target word \(x\), the context consists of the words that occur within a distance \(d\) on either side of each occurrence of \(x\).

For a target word at position \(i\), a neighboring word at position \(j\) is included when:

\[
0 < |i-j| \leq d
\]

This excludes the target word itself while including words up to \(d\) positions before and after it.

In [5]:
# Code Block 4: Extract Context Words Within Distance d

def extract_context_words(sentences, target_word, d):
    context_words = []

    for sentence in sentences:
        for i, word in enumerate(sentence):

            if word == target_word:
                start = max(0, i - d)
                end = min(len(sentence), i + d + 1)

                for j in range(start, end):
                    if j != i:
                        context_words.append(sentence[j])

    return context_words


# Test the function
target_word = "dog"
distance = 2

dog_context = extract_context_words(
    normalized_sentences,
    target_word,
    distance
)

print("Target word   :", target_word)
print("Distance (d)  :", distance)
print("Context words :", dog_context)

Target word   : dog
Distance (d)  : 2
Context words : ['the', 'chased', 'the', 'the', 'played', 'with', 'the', 'chased', 'the']


## 5. Construct the Distributional Vector

A distributional vector represents the target word using the frequencies of words that appear in its surrounding context.

For each unique context word, one vector dimension is created. The value in that dimension is the number of times the context word occurs within distance \(d\) of the target word.

For the target word \(x\), the value associated with context word \(w\) can be written as:

\[
V_x(w) =
\sum_{i:x_i=x}
\sum_{j:0<|i-j|\leq d}
I(x_j=w)
\]

where \(I(x_j=w)\) equals 1 when the context word at position \(j\) is \(w\), and 0 otherwise.

In [6]:
# Code Block 5: Build a Distributional Vector

def build_distributional_vector(context_words):
    context_counts = Counter(context_words)

    # Use alphabetical order so the vector dimensions are deterministic
    context_vocabulary = sorted(context_counts.keys())

    vector = [
        context_counts[word]
        for word in context_vocabulary
    ]

    return context_vocabulary, vector


dog_vocabulary, dog_vector = build_distributional_vector(dog_context)

print("Context vocabulary :", dog_vocabulary)
print("Distribution vector:", dog_vector)

Context vocabulary : ['chased', 'played', 'the', 'with']
Distribution vector: [2, 1, 5, 1]


## 6. Concrete Example: Representing "dog" as a Distributional Vector

For the concrete example, the target word is **dog** and the context-window distance is \(d=2\).

Every occurrence of `dog` is examined in the normalized corpus. All words located within two positions before or after the target word are collected, excluding the target word itself.

The frequencies of these context words form the values of the final distributional vector.

In [7]:
# Code Block 6: Concrete Distributional Representation Example

target_word = "dog"
distance = 2

context_words = extract_context_words(
    normalized_sentences,
    target_word,
    distance
)

context_counts = Counter(context_words)

context_vocabulary, distribution_vector = build_distributional_vector(
    context_words
)

print(f"Target word: {target_word}")
print(f"Context distance (d): {distance}")

print("\nOccurrences of the target word:")
for i, sentence in enumerate(normalized_sentences, start=1):
    if target_word in sentence:
        print(f"Sentence {i}: {sentence}")

print("\nExtracted context words:")
print(context_words)

print("\nContext word frequencies:")
for word in context_vocabulary:
    print(f"{word:<10} -> {context_counts[word]}")

print("\nContext vocabulary:")
print(context_vocabulary)

print("\nFinal distributional vector:")
print(distribution_vector)

print(
    f"\nTherefore, '{target_word}' is represented as "
    f"{distribution_vector} using the context dimensions "
    f"{context_vocabulary}."
)

Target word: dog
Context distance (d): 2

Occurrences of the target word:
Sentence 1: ['the', 'dog', 'chased', 'the', 'cat']
Sentence 2: ['the', 'dog', 'played', 'with', 'the', 'ball']
Sentence 5: ['the', 'dog', 'chased', 'the', 'mouse']

Extracted context words:
['the', 'chased', 'the', 'the', 'played', 'with', 'the', 'chased', 'the']

Context word frequencies:
chased     -> 2
played     -> 1
the        -> 5
with       -> 1

Context vocabulary:
['chased', 'played', 'the', 'with']

Final distributional vector:
[2, 1, 5, 1]

Therefore, 'dog' is represented as [2, 1, 5, 1] using the context dimensions ['chased', 'played', 'the', 'with'].
